# 01 — Exploratory Data Analysis
**Fraud Detection & Anomaly Scoring System**

Goals:
- Understand the class imbalance problem
- Explore feature distributions
- Identify patterns that distinguish fraud from legitimate transactions

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})
sns.set_theme(style='darkgrid', palette='muted')

from src.config import RAW_CSV
from src.features import load_raw

df = load_raw()
df.head()

## 1. Dataset Overview

In [ ]:
print(f"Shape        : {df.shape}")
print(f"Memory       : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"Missing vals : {df.isnull().sum().sum()}")
print(f"Fraud count  : {df['Class'].sum():,} ({df['Class'].mean():.4%})")
print(f"Legit count  : {(df['Class']==0).sum():,}")
df.describe().T

## 2. Class Imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df['Class'].value_counts()
axes[0].bar(['Legitimate', 'Fraud'], counts.values, color=['#27ae60','#e74c3c'], edgecolor='white')
axes[0].set_title('Transaction Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1000, f'{v:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(counts.values, labels=['Legitimate','Fraud'], autopct='%1.3f%%',
            colors=['#27ae60','#e74c3c'], startangle=90)
axes[1].set_title('Class Split (Pie)')

plt.suptitle('Severe Class Imbalance — Only 0.17% Fraud', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_class_imbalance.png', dpi=120)
plt.show()

## 3. Transaction Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls, label, color in [(0,'Legitimate','#2ecc71'), (1,'Fraud','#e74c3c')]:
    subset = df[df['Class']==cls]['Amount']
    axes[0].hist(subset, bins=60, alpha=0.6, color=color, label=label)

axes[0].set_title('Amount Distribution (raw)')
axes[0].set_xlabel('Amount ($)')
axes[0].legend()
axes[0].set_xlim(0, 1000)

for cls, label, color in [(0,'Legitimate','#2ecc71'), (1,'Fraud','#e74c3c')]:
    subset = np.log1p(df[df['Class']==cls]['Amount'])
    axes[1].hist(subset, bins=60, alpha=0.6, color=color, label=label)

axes[1].set_title('Amount Distribution (log1p transformed)')
axes[1].set_xlabel('log1p(Amount)')
axes[1].legend()

plt.suptitle('Fraud vs Legitimate — Transaction Amounts', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_amount_distribution.png', dpi=120)
plt.show()

print("Amount statistics by class:")
df.groupby('Class')['Amount'].describe().round(2)

## 4. Transaction Time Patterns

In [ ]:
df['Hour'] = (df['Time'] % 86400) // 3600

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fraud by hour
fraud_by_hour = df[df['Class']==1].groupby('Hour').size()
legit_by_hour = df[df['Class']==0].groupby('Hour').size()
fraud_rate_by_hour = (fraud_by_hour / (fraud_by_hour + legit_by_hour.reindex(fraud_by_hour.index, fill_value=1))) * 100

axes[0].bar(fraud_by_hour.index, fraud_by_hour.values, color='#e74c3c', alpha=0.8)
axes[0].set_title('Fraud Transactions by Hour of Day')
axes[0].set_xlabel('Hour (0-23)')
axes[0].set_ylabel('Fraud Count')

axes[1].plot(fraud_rate_by_hour.index, fraud_rate_by_hour.values, marker='o', color='#e74c3c', lw=2)
axes[1].fill_between(fraud_rate_by_hour.index, fraud_rate_by_hour.values, alpha=0.2, color='#e74c3c')
axes[1].set_title('Fraud Rate (%) by Hour of Day')
axes[1].set_xlabel('Hour (0-23)')
axes[1].set_ylabel('Fraud Rate (%)')

plt.suptitle('Time-of-Day Fraud Patterns', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_time_patterns.png', dpi=120)
plt.show()

## 5. PCA Feature Distributions (Top Discriminating Features)

In [ ]:
# Known high-signal features from literature: V14, V10, V12, V4, V11
top_features = ['V14', 'V10', 'V12', 'V4', 'V11', 'V17']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    fraud_vals = df[df['Class']==1][feat]
    legit_vals = df[df['Class']==0][feat]
    
    axes[i].hist(legit_vals, bins=80, alpha=0.5, color='#27ae60', label='Legitimate', density=True)
    axes[i].hist(fraud_vals, bins=80, alpha=0.7, color='#e74c3c', label='Fraud', density=True)
    axes[i].set_title(f'{feat} Distribution')
    axes[i].legend(fontsize=8)
    axes[i].set_xlim(-10, 10)

plt.suptitle('Top Discriminating PCA Features: Fraud vs Legitimate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_feature_distributions.png', dpi=120)
plt.show()

## 6. Correlation Heatmap

In [ ]:
corr_features = [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']
corr_matrix = df[corr_features].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=False, cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, ax=ax,
    linewidths=0.3, cbar_kws={'shrink': .8}
)
ax.set_title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_correlation_heatmap.png', dpi=120)
plt.show()

## 7. Correlation with Target (Class)

In [ ]:
corr_with_target = df[[f'V{i}' for i in range(1,29)] + ['Amount']].corrwith(df['Class']).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e74c3c' if v < 0 else '#27ae60' for v in corr_with_target]
ax.bar(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='white')
ax.axhline(0, color='black', lw=0.8)
ax.set_title('Feature Correlation with Target (Class)', fontsize=13, fontweight='bold')
ax.set_xlabel('Feature')
ax.set_ylabel('Pearson Correlation')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../reports/eda_target_correlation.png', dpi=120)
plt.show()

print("\nTop 10 features correlated with fraud:")
print(corr_with_target.abs().nlargest(10))

## 8. Box Plots — Top Features by Class

In [ ]:
top8 = corr_with_target.abs().nlargest(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(top8):
    df.boxplot(column=feat, by='Class', ax=axes[i], 
               boxprops=dict(color='#2c3e50'),
               medianprops=dict(color='#e74c3c', lw=2))
    axes[i].set_title(feat)
    axes[i].set_xlabel('Class (0=Legit, 1=Fraud)')
    axes[i].set_xticklabels(['Legitimate', 'Fraud'])

plt.suptitle('Top Discriminating Features — Box Plots by Class', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/eda_boxplots.png', dpi=120)
plt.show()

## 9. EDA Summary

| Finding | Detail |
|---------|--------|
| Class imbalance | 0.172% fraud — extreme imbalance, SMOTE required |
| Amount | Fraud transactions have lower median amounts than legitimate ones |
| Time | Fraud spikes in off-peak hours (2am–4am) |
| Top features | V14, V10, V12, V4, V11 show strongest class separation |
| Correlations | PCA features are uncorrelated by design |

> **Next step:** Feature Engineering notebook (`02_feature_engineering.ipynb`)